In [ ]:
# ---- EXPERIMENT CONFIG: edit this cell for each run --------------------------
# Pose encoding of the GATr Clifford flow on SYMSOL I images (10 % of the training data): the harm6 + nearest-coupling
# recipe of reports/symsol_stf/README.md (section 7, "harm6 + NC"), with only the pose encoding changed.
#   harm6_nc     the reference: spherecloud condition tokens, P1 pose (frame_pin), and every condition token gets the
#                49 real spherical harmonics Y_lm, l <= 6, of its direction u in the current pose's frame
#                (--gatr_pose_harmonics 6), + --symsol_nearest_coupling.
#   fourier3_nc  the same, but u enters through the IPDF positional encoding (Murphy et al., ICML 2021):
#                sin(2^k u_j), cos(2^k u_j), k = 0, 1, 2, j = 1..3, 18 scalars per token (--gatr_pose_fourier 3;
#                IPDF uses 3 frequencies on SYMSOL). Everything else is identical.
# Protocol (as harm6 / harm6_nc): 4,500 of 45,000 training views per shape (22,500 images), all 5,000 test views per
# shape, scored by the rotation error to the NEAREST EQUIVALENT pose; ResNet-50, no augmentation, no EMA, AdamW lr 1e-4
# x sqrt(64/32), step schedule (x0.1 every 15 epochs) without warmup, global batch 64, 40 epochs, no final flow
# log-likelihood (--no-symsol_likelihood).
# HW: 'rtx' = one RTX Pro 6000, one process (as the harm6_nc RTX run, 145 s / epoch); 'l4x4' = 4 x L4, one process per
# GPU with 16 images each and SyncBatchNorm (as the harm6_nc 4 x L4 run, 386 s / epoch).
# ETA per run: ~1.8 h on 'rtx', ~4.5 h on 'l4x4' (and 2x quota). Over the 30-minute limit of CLAUDE.md: a human
# approves the launch. RUN ONE VARIANT PER KERNEL.
#
# OFFLINE (RTX / L4 have no internet) attach: a repo snapshot carrying --gatr_pose_fourier (pose3d/scripts/make_snapshot.py),
#   syfry5suvzovvakmuj/ga-research-offline-deps, syfry5suvzovvakmuj/ga-research-offline-deps-py313,
#   syfry5suvzovvakmuj/ga-research-gatr-deps, syfry5suvzovvakmuj/symsol1-10pct-fulltest.
RUN = 'fourier3_nc'
HW = 'rtx'
WANDB_API_KEY = None
REPO = 'chagrygoris/GA-Research'
BRANCH = 'claude/jolly-mayer-xhjtgm'
COMMON = ['--model', 'clifford_flow', '--encoder', 'resnet50', '--dataset', 'symsol', '--symsol_set', '1',
          '--symsol_train_views', '4500', '--no-use_warp', '--no-use_synth', '--no-ema', '--fixed_val_noise',
          '--batch_size', '64', '--n_epochs', '40', '--optimizer', 'adamw', '--lr', '1e-4', '--lr_scaling', 'sqrt',
          '--lr_schedule', 'step', '--warmup_epochs', '0', '--num_workers', '4', '--no-symsol_likelihood',
          '--vector_field', 'gatr', '--cond_tokens', 'spherecloud', '--pose_tokens', 'frame_pin',
          '--condition_head', 'none', '--dircloud_scalars', '64', '--gatr_mv_channels', '8', '--gatr_s_channels', '32',
          '--no-gatr_zero_readout', '--symsol_nearest_coupling']
COMMON += {'rtx': ['--no-ddp'], 'l4x4': ['--ddp', '--sync_bn']}[HW]
VARIANTS = {
    'symsol1_10pct_gatr_spherecloud_harm6_nc': ['--gatr_pose_harmonics', '6'],
    'symsol1_10pct_gatr_spherecloud_fourier3_nc': ['--gatr_pose_fourier', '3'],
}
SHORT = {'harm6_nc': 'symsol1_10pct_gatr_spherecloud_harm6_nc', 'fourier3_nc': 'symsol1_10pct_gatr_spherecloud_fourier3_nc'}
assert RUN in SHORT, f'RUN must be one of {list(SHORT)}'
SNAPSHOT_MARKER = 'gatr_pose_fourier'   # a repo snapshot / clone must carry the IPDF pose encoding
SNAPSHOT_FILE = 'models/clifford_flow.py'
ARGS = COMMON


In [ ]:
import glob, os, shutil, socket, subprocess, tarfile

def online(host='github.com', timeout=5):
    try:
        socket.create_connection((host, 443), timeout=timeout).close()
        return True
    except OSError:
        return False

ONLINE = online()
print('internet:', ONLINE)
if os.path.exists('/kaggle/working/GA-Research'):
    shutil.rmtree('/kaggle/working/GA-Research')
os.chdir('/kaggle/working')
markers = glob.glob('/kaggle/input/**/pose3d/pose3d/config.py', recursive=True)
tgzs = glob.glob('/kaggle/input/**/ga-research-*.tar.gz', recursive=True)
if markers:        # a repo snapshot dataset, auto-extracted by Kaggle
    assert len(markers) == 1, f'attach exactly one repo snapshot, found {markers}'
    shutil.copytree(markers[0][:markers[0].index('/pose3d/pose3d/config.py')], '/kaggle/working/GA-Research')
elif tgzs:
    assert len(tgzs) == 1, f'attach exactly one repo snapshot, found {tgzs}'
    with tarfile.open(tgzs[0]) as t:
        t.extractall('/kaggle/working')
else:
    assert ONLINE, 'no internet and no repo snapshot attached'
    r = subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, f'https://github.com/{REPO}.git'],
                       capture_output=True, text=True)
    print(r.stdout + r.stderr); r.check_returncode()
print(subprocess.run(['git', '-C', 'GA-Research', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)
src = open(f'/kaggle/working/GA-Research/pose3d/pose3d/{SNAPSHOT_FILE}').read()
assert SNAPSHOT_MARKER in src, f'stale code: {SNAPSHOT_MARKER!r} is not in {SNAPSHOT_FILE}'


In [ ]:
# Dependencies Kaggle lacks. Online: PyPI / GitHub, with the image's torch pinned by a constraints file.
# Offline: the wheels of the ga-research-offline-deps and ga-research-gatr-deps datasets (installed without
# dependencies: the image's torch must stay).
import importlib, importlib.metadata as md, sys
# ga-research-offline-deps holds the wheels and ResNet weights; ga-research-offline-deps-py313 adds the healpy
# wheel for the Python 3.13 image (October 2026). Every ga-research-offline-deps* folder attached is read.
deps = sorted(glob.glob('/kaggle/input/**/ga-research-offline-deps*', recursive=True))
gatr_deps = glob.glob('/kaggle/input/**/ga-research-gatr-deps', recursive=True)
if deps:
    wheels = sorted(w for d in deps for w in glob.glob(os.path.join(d, '*.whl')))
    if gatr_deps:
        wheels += sorted(glob.glob(os.path.join(gatr_deps[0], '*.whl')))
    # Keep only the wheels this image's Python can run: the datasets may hold one healpy build per Python version
    # (the Kaggle image moved from 3.12 to 3.13 in October 2026), and pip rejects the whole command on one bad wheel.
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    supported = set(sys_tags())
    wheels = [w for w in wheels if parse_wheel_filename(os.path.basename(w))[3] & supported]
    print('wheels:', [os.path.basename(w) for w in wheels])
    assert any('healpy' in w for w in wheels), 'no healpy wheel for this Python: attach a ga-research-offline-deps* dataset that has one'
    subprocess.run(['pip', 'install', '-q', '--no-index', '--no-deps', '--disable-pip-version-check', *wheels], check=True)
    ckpt_dir = os.path.expanduser('~/.cache/torch/hub/checkpoints'); os.makedirs(ckpt_dir, exist_ok=True)
    for w in glob.glob('/kaggle/input/**/resnet*-*.pth', recursive=True):   # ImageNet weights (resnet50, resnet101)
        shutil.copy(w, os.path.join(ckpt_dir, os.path.basename(w)))
else:
    assert ONLINE, 'no internet and no ga-research-offline-deps attached'
    cons = []
    for p in ['torch', 'torchvision', 'numpy', 'scipy', 'pillow', 'tqdm', 'wandb', 'scikit-image', 'matplotlib',
              'pandas', 'sympy', 'astropy', 'packaging']:
        try:
            cons.append(f'{p}=={md.version(p)}')
        except md.PackageNotFoundError:
            pass
    open('/kaggle/working/constraints.txt', 'w').write('\n'.join(cons) + '\n')
    subprocess.run(['pip', 'install', '-q', '-c', '/kaggle/working/constraints.txt', 'e3nn==0.5.9', 'healpy==1.19.0',
                    'git+https://github.com/chagrygoris/image2sphere.git',
                    'git+https://github.com/chagrygoris/clifford-group-equivariant-neural-networks.git'], check=True)
    # GATr, the vector field: --no-deps, its pins would replace torch.
    subprocess.run(['pip', 'install', '-q', '--no-deps', 'einops', 'opt_einsum',
                    'git+https://github.com/Qualcomm-AI-research/geometric-algebra-transformer.git'], check=True)
for mod in ['e3nn', 'healpy', 'image2sphere', 'clifford']:
    importlib.import_module(mod)
# GATr hard-imports xformers (only for attention masks the flow never passes); the repo installs a stub before
# importing it (models/gatr_denoiser._import_gatr), so check the import the way the training process does.
sys.path.insert(0, '/kaggle/working/GA-Research/pose3d')
from pose3d.models.gatr_denoiser import _import_gatr
_import_gatr()
print('python', sys.version.split()[0], '| deps import cleanly')


In [ ]:
# W&B: live when online (the wandb_api_key secret, or WANDB_API_KEY above); offline files + WANDB_SYNC lines otherwise.
key = WANDB_API_KEY
if key is None and ONLINE:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('wandb_api_key')
    except Exception as e:   # no secret attached: fall back to offline logging
        print('wandb_api_key secret not available:', e)
if key and ONLINE:
    import wandb
    wandb.login(key=key)
    os.environ['WANDB_API_KEY'] = key
else:
    os.environ['WANDB_MODE'] = 'offline'
print('wandb mode:', os.environ.get('WANDB_MODE', 'online'))


In [ ]:
os.chdir('/kaggle/working/GA-Research/pose3d')   # the package is found from here, no install needed
found = glob.glob('/kaggle/input/**/symsol/test/rotations.npz', recursive=True)
assert found, 'attach the SYMSOL dataset (symsol/{train,test}/rotations.npz + images/)'
DATA = found[0][:-len('/symsol/test/rotations.npz')]
print('SYMSOL root:', DATA, flush=True)
name = SHORT[RUN]
cmd = [sys.executable, '-u', '-m', 'pose3d', f'--path_to_datasets={DATA}', '--platform=kaggle', f'--run_name={name}',
       *COMMON, *VARIANTS[name]]
print(' '.join(cmd), flush=True)
# The output is also written to /kaggle/working/<run_name>.log: Kaggle keeps /kaggle/working as the kernel's output
# even when its own log stream comes back empty (seen on long RTX sessions), so the per-epoch metrics survive.
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout, flush=True)
with open(f'/kaggle/working/{name}.log', 'w') as fh:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ})
    for line in proc.stdout:
        print(line, end='', flush=True)
        fh.write(line); fh.flush()
    rc = proc.wait()
# a crash marks the Kaggle run as failed instead of passing silently.
assert rc == 0, f'pose3d exited with code {rc}'
